# The nine Cohere models on vLLM, measured

This notebook measures the vLLM column of `docs/model_backends.md` (#52). Until now that column was read off vLLM's model registry, not run. It is the companion of `colab_sglang_cohere_matrix.ipynb`: same models, same probes, same pass rule, same results format.

Each model gets five checks:

| Check | What it records |
|---|---|
| **registry** | Whether this vLLM registers the model's architecture. This is the reading the doc has today. |
| **offline** | AuditKIT's `vllm:` backend (the in-process `LLM` engine), in its **own Python process**, so one model's leftovers can't fail the next. If loading fails, the reason is classified: *removed from vLLM*, *no implementation*, *transformers too old*, *out of memory*, *download or gating*, or *other*, with the message. |
| **offline text** | The same five-language probes and pass rule as the `hf:` column: PASS at 4/5 or better, PARTIAL for 1–3, FAIL for 0. It also records whether the Cohere chat template was applied, and how much GPU memory `unload()` returned. Those check two fixes in #54 on real hardware. |
| **serve** | `vllm serve` with the same model, reached through `api:`, plus the same probes. |
| **image** | Vision architectures only, if `vllm serve` loads them: a red square, asked "what colour is it?" through `Sample.images`. Offline `vllm:` doesn't take images. |

**RTX PRO 6000 / RTX 50-series (SM 12.x) with Colab's CUDA 12.8:** vLLM samples with FlashInfer by default, and FlashInfer compiles for SM 12.x only with CUDA ≥ 12.9, so every model failed to load with "FlashInfer requires GPUs with sm75 or higher". AuditKIT's `vllm:` backend switches vLLM to its own sampler on such a machine by itself, and this notebook passes the same setting to `vllm serve` (`compat.py --vllm-launch`, printed in the install cell). Elsewhere, or with CUDA ≥ 12.9, nothing changes, and greedy (temperature 0) answers don't depend on the sampler.

**Disk:** the 32B downloads are ~62 GB each. Pip and Xet caches are cleared before the sweep, the disk each model needs is read from its real file sizes on the Hub, and each model's weights are deleted after it's tested.

**Layout check** (vision models): a picture that is red on the left and blue on the right, asked for each side's colour, the same check the SGLang notebooks run.

**Runtime:** 80 GB or more runs all nine. On 40 GB, the two 32B models are SKIP. After testing, each model's weights are deleted from the disk cache.

**Tokens:** the notebook asks for them in hidden boxes, and they are never saved.
- `GITHUB_TOKEN`: installs AuditKIT.
- `HF_TOKEN`: all nine models are gated, so accept their licences first.

**Time:** about 2 to 3 hours.

In [ ]:
# ---- settings ----
BRANCH = "fix/vllm-sm12-flashinfer-v2"   # vLLM on SM 12.x with CUDA < 12.9 (FlashInfer sampler off). "main" once merged
REPO_URL = "github.com/aryapratinavseth/AuditKIT-Internal.git"   # this repo: see #51 for why not another copy
PORT = 30000
DELETE_AFTER = True
LOAD_TIMEOUT_S = 3600            # includes the download
TRANSFORMERS_BACKEND_RETRY = True   # a model vLLM can't serve natively (Aya Vision: removed after v0.24.0) is
                                    # retried through vLLM's Transformers backend; labelled non-default
# (repo, architecture, min GPU GB in bf16, is_vision_arch)
MODELS = [
    ("CohereLabs/tiny-aya-global", "cohere2", 8,  False),
    ("CohereLabs/tiny-aya-fire",   "cohere2", 8,  False),
    ("CohereLabs/tiny-aya-water",  "cohere2", 8,  False),
    ("CohereLabs/tiny-aya-earth",  "cohere2", 8,  False),
    ("CohereLabs/North-Micro-Vision-Instruct", "cohere_compass", 6, True),
    ("CohereLabs/aya-expanse-8b",  "cohere", 18, False),
    ("CohereLabs/aya-vision-8b",   "aya_vision", 20, True),
    ("CohereLabs/aya-expanse-32b", "cohere", 70, False),
    ("CohereLabs/aya-vision-32b",  "aya_vision", 75, True),
]

# `expect` is any one acceptable answer. Asking is not enough: a model that loads
# and answers in English scores the same as one that understood the question.
PROBES = [
    ("fr", "Quel est le plus grand port de France ? Un mot.",
     ("Marseille", "marseille", "Marseill")),
    ("hi", "\u092d\u093e\u0930\u0924 \u0915\u0940 \u0930\u093e\u091c\u0927\u093e\u0928\u0940 \u0915\u094d\u092f\u093e \u0939\u0948? \u090f\u0915 \u0936\u092c\u094d\u0926\u0964",
     ("\u0926\u093f\u0932\u094d\u0932\u0940", "\u0928\u090f\u0940 \u0926\u093f\u0932\u094d\u0932\u0940", "delhi")),
    ("zh", "\u4e2d\u56fd\u7684\u9996\u90fd\u662f\u54ea\u91cc\uff1f\u7528\u4e00\u4e2a\u8bcd\u3002",
     ("\u5317\u4eac", "beijing")),
    ("ar", "\u0645\u0627 \u0647\u064a \u0639\u0627\u0635\u0645\u0629 \u0645\u0635\u0631\uff1f \u0643\u0644\u0645\u0629 \u0648\u0627\u062d\u062f\u0629.",
     ("\u0627\u0644\u0642\u0627\u0647\u0631\u0629", "cairo")),
    ("th", "\u0e40\u0e21\u0e37\u0e2d\u0e07\u0e2b\u0e25\u0e27\u0e07\u0e02\u0e2d\u0e07\u0e1b\u0e23\u0e30\u0e40\u0e17\u0e28\u0e44\u0e17\u0e22\u0e04\u0e37\u0e2d\u0e2d\u0e30\u0e32\u0e2b\u0e48\u0e2d\u0e04\u0e23\u0e31\u0e1a? \u0e2b\u0e19\u0e34\u0e48\u0e07\u0e04\u0e33",
     ("\u0e01\u0e23\u0e38\u0e07\u0e40\u0e17\u0e1e", "bangkok")),
]
# "กทม." is the standard abbreviation of Bangkok: a correct answer the #49 probe list misses
PROBES = [(c, p, tuple(e) + (("กทม",) if c == "th" else ())) for c, p, e in PROBES]
RESULTS = []

In [ ]:
# ---- tokens: paste into the boxes (hidden input; never saved in the notebook) ----
import os
from getpass import getpass
for key, why in (("GITHUB_TOKEN", "required: installs AuditKIT"),
                 ("HF_TOKEN", "required: the Cohere models are gated")):
    if os.environ.get(key):
        print(f"{key}: already set in this runtime"); continue
    value = getpass(f"Paste {key} ({why}): ").strip()
    if value:
        os.environ[key] = value
print({k: bool(os.environ.get(k)) for k in ("GITHUB_TOKEN", "HF_TOKEN")})

In [ ]:
# ---- install AuditKIT with vLLM, then check the install is what it should be (#51) ----
import json, subprocess, sys
def sh(*a, **kw):
    return subprocess.run(a, capture_output=True, text=True, **kw)

if "torch" in sys.modules:
    raise SystemExit("torch was imported before this install: Runtime -> Restart session, then run from the top.")
REPO = "/content/AuditKIT-Internal"
url = f"https://{os.environ['GITHUB_TOKEN']}@{REPO_URL}"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "--branch", BRANCH, url, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "-q", url, BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "-q", "-B", BRANCH, "FETCH_HEAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[vllm,requests]"], check=True)
# vLLM brings a CUDA 13 torch; Colab's CUDA 12.8 torchaudio would then break every transformers import
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"], check=False)
sys.path.insert(0, f"{REPO}/src")

VERSIONS = {p: sh(sys.executable, "-c", f"import importlib.metadata as m; print(m.version('{p}'))").stdout.strip() or None
            for p in ("auditkit", "vllm", "transformers", "torch")}
VERSIONS["commit"] = sh("git", "-C", REPO, "log", "--oneline", "-1").stdout.strip()
print(VERSIONS)
# #51: a stale AuditKIT (1.0.0 extras) resolves transformers 4.57.6, and North then fails for the wrong reason
assert (VERSIONS["transformers"] or "").startswith("5."), f"transformers {VERSIONS['transformers']}: need 5.x (see #51)"
ak_v = tuple(int(x) for x in (VERSIONS["auditkit"] or "0.0").split(".")[:2])
assert ak_v >= (1, 2), f"auditkit {VERSIONS['auditkit']}: not this repo's code (see #51)"

import torch
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
print(GPU_NAME, f"{GPU_GB:.0f} GB")
reg = sh(sys.executable, "-c",
          "import json; from vllm import ModelRegistry; print(json.dumps(sorted(ModelRegistry.get_supported_archs())))")
try:
    REGISTERED = set(json.loads(reg.stdout.strip().splitlines()[-1]))
except Exception:
    REGISTERED = set()
    print("could not read vLLM's registry:", (reg.stderr or reg.stdout)[-500:])
print(len(REGISTERED), "architectures registered in this vLLM")

## Helpers

- **The offline worker:** a small script that loads one model through AuditKIT's `vllm:` backend in its own process and writes a JSON result.
- **The server helpers:** start `vllm serve`, wait until healthy, stop it.
- **Classification:** reads vLLM's own error messages.

In [ ]:
import glob, json, re, shutil, signal, time
import requests
from huggingface_hub import hf_hub_download, scan_cache_dir
from auditkit.model import Request
from auditkit.model.api_gen import APIModel

WORKER_PY = "/content/vllm_one.py"
open(WORKER_PY, "w").write('"""One model through AuditKIT\'s offline vllm: backend, in its own process: python vllm_one.py REPO OUT.json"""\nimport json, sys, time, traceback\nrepo, out = sys.argv[1], sys.argv[2]\nPROBES = json.loads(sys.argv[3])\nENGINE = json.loads(sys.argv[4]) if len(sys.argv) > 4 else {}     # e.g. {"model_impl": "transformers"}\nres = {"repo": repo, "engine": ENGINE}\ntry:\n    import torch\n    from auditkit.model import Request\n    from auditkit.model.vllm_gen import VLLMModel\n    t0 = time.time()\n    # dtype auto: vLLM keeps bf16 where the GPU has it and casts to fp16 where it doesn\'t (T4, V100)\n    m = VLLMModel(repo, name=f"vllm:{repo}", dtype="auto", gpu_memory_utilization=0.85,\n                  max_model_len=4096, trust_remote_code=True, **ENGINE)\n    m._ensure_llm()\n    res["load_s"] = round(time.time() - t0)\n    # the named-template fix (#54): the Cohere chat template must be applied, not the raw prompt\n    res["chat_template"] = m._chat_tokenizer(Request(prompt="x")) is not None\n    right, answers = 0, {}\n    for code_, prompt, expect in PROBES:\n        g = m.generate([Request(prompt=prompt, params={"max_tokens": 32, "temperature": 0.0})])[0].completions[0]\n        text = (g.text or "").strip().replace("\\n", " ") or f"EMPTY (finish={g.finish_reason})"\n        answers[code_] = text[:70]\n        right += any(e in text for e in expect)\n    res.update(right=right, answers=answers)\n    free = lambda: torch.cuda.mem_get_info()[0] / 2**30\n    before = free()\n    m.unload()                                   # the teardown fix (#54): the memory must come back\n    res.update(free_before_unload_gb=round(before, 1), free_after_unload_gb=round(free(), 1),\n               total_gb=round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))\nexcept BaseException as e:\n    res["error"] = f"{type(e).__name__}: {e}"\n    res["traceback"] = traceback.format_exc()[-3000:]\njson.dump(res, open(out, "w"), ensure_ascii=False)\n')

def record(model, check, outcome, detail=""):
    RESULTS.append({"model": model, "check": check, "outcome": outcome, "detail": str(detail)[:600]})
    print(f"  [{outcome:8}] {model.split('/')[-1]:28s} {check:12s} {str(detail)[:150]}", flush=True)

def config_archs(repo):
    cfg = json.load(open(hf_hub_download(repo, "config.json", token=os.environ.get("HF_TOKEN"))))
    return cfg.get("architectures") or [], cfg.get("model_type")

CAUSES = [  # vLLM's own messages first; first match wins
    ("removed from vLLM", r"was supported in vLLM until v[\d.]+, and is not supported anymore"),
    ("plugin only", r"is not supported in-tree anymore"),
    ("no implementation", r"Model architectures? .* (are|is) not supported"),
    ("transformers too old", r"(does not recognize this architecture|but Transformers does not recognize)"),
    ("kernel build", r"(FlashInfer requires GPUs with sm|SM 12\.x requires CUDA|No supported CUDA architectures)"),
    ("out of memory", r"(out of memory|OutOfMemory|No available memory for the cache blocks|Free memory on device)"),
    ("download or gating", r"(GatedRepoError|401 Client Error|403 Client Error|RepositoryNotFound)"),
]

def classify(text):
    for cause, pattern in CAUSES:
        m = re.search(pattern, text)
        if m:
            line = next((l for l in text.splitlines() if m.group(0) in l), m.group(0))
            return cause, line.strip()
    errors = [l for l in text.splitlines() if re.search(r"(Error|Exception|error:)", l)]
    last = errors[-1] if errors else (text.strip().splitlines() or ["empty log"])[-1]
    return "other", last.strip()

def verdict(right):
    return "PASS" if right >= 4 else ("PARTIAL" if right else "FAIL")

def free_gb():
    out = sh("nvidia-smi", "--query-gpu=memory.free", "--format=csv,noheader,nounits").stdout.strip()
    return float(out.splitlines()[0]) / 1024 if out else 0.0

def wait_memory_back():
    for _ in range(60):
        if free_gb() > GPU_GB - 2: return
        time.sleep(2)

SERVER_ENV = {k: v for k, v in os.environ.items() if k != "VLLM_ENABLE_V1_MULTIPROCESSING"}
# what vLLM needs on THIS GPU (compat.py --vllm-launch): on an SM 12.x GPU with CUDA < 12.9, FlashInfer can't
# build, so vLLM samples with its own sampler. The offline vllm: backend applies the same advice by itself.
_adv = sh(sys.executable, f"{REPO}/src/auditkit/compat.py", "--vllm-launch", "x").stdout
_line = next((l for l in _adv.splitlines() if l.startswith("vllm launch advice:")), None)
LAUNCH_ADVICE = json.loads(_line.split(":", 1)[1]) if _line else {"env": {}, "notes": ["compat.py has no --vllm-launch"]}
SERVER_ENV.update(LAUNCH_ADVICE["env"])
VERSIONS["launch_advice"] = LAUNCH_ADVICE
print("vLLM launch advice:", LAUNCH_ADVICE)

NATIVE_ONLY = ("removed from vLLM", "no implementation", "plugin only")   # causes the Transformers backend may fix
TF_LABEL = " [non-default: vLLM's Transformers backend]"

def start_server(repo, log, extra=()):
    return subprocess.Popen([sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", repo,
                             "--port", str(PORT), "--max-model-len", "4096", "--gpu-memory-utilization", "0.85",
                             "--trust-remote-code", *extra],
                            stdout=open(log, "w"), stderr=subprocess.STDOUT, start_new_session=True, env=SERVER_ENV)

def wait_healthy(proc, timeout_s):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if proc.poll() is not None:
            return False
        try:
            if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=5).ok:
                return True
        except requests.RequestException:
            pass
        time.sleep(10)
    return False

def stop(proc):
    try:
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM); proc.wait(timeout=120)
    except Exception:
        try: os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
        except Exception: pass
    wait_memory_back()

def probes(model):
    right, answers = 0, {}
    for code_, prompt, expect in PROBES:
        g = model.generate([Request(prompt=prompt, params={"messages": [{"role": "user", "content": prompt}],
                                                            "max_tokens": 32, "temperature": 0.0})])[0].completions[0]
        text = (f"ERR {g.error}" if g.finish_reason == "error" else
                (g.text or "").strip().replace("\n", " ") or f"EMPTY (finish={g.finish_reason})")[:70]
        answers[code_] = text
        right += any(e in text for e in expect)
    return right, answers

def image_check(model):
    from PIL import Image
    img = Image.new("RGB", (224, 224), "white")
    img.paste((220, 20, 20), (56, 56, 168, 168))
    q = "What colour is the square in the picture? Answer with one word."
    g = model.generate([Request(prompt=q, params={"messages": [{"role": "user", "content": q}], "images": [img],
                                                  "max_tokens": 16, "temperature": 0.0})])[0].completions[0]
    return (g.text or g.error or "").strip()[:80]

def delete_weights(repo):
    for r in scan_cache_dir().repos:
        if r.repo_id == repo:
            shutil.rmtree(r.repo_path, ignore_errors=True)

def download_gb(repo):
    # what vLLM will download: the weights plus the small files
    from huggingface_hub import HfApi
    info = HfApi().model_info(repo, files_metadata=True, token=os.environ.get("HF_TOKEN"))
    return sum((s.size or 0) for s in info.siblings) / 2**30

def free_disk_gb():
    from huggingface_hub import constants
    os.makedirs(constants.HF_HUB_CACHE, exist_ok=True)
    return shutil.disk_usage(constants.HF_HUB_CACHE).free / 2**30

def clear_caches():
    # pip keeps every wheel it installed (vLLM and torch are GBs); huggingface_hub's Xet chunk cache too
    subprocess.run([sys.executable, "-m", "pip", "cache", "purge"], capture_output=True)
    from huggingface_hub import constants
    shutil.rmtree(os.path.join(constants.HF_HOME, "xet"), ignore_errors=True)

def layout_check(model):
    # left half red, right half blue: right only if the image tokens keep their positions
    from PIL import Image
    img = Image.new("RGB", (448, 224), (20, 40, 220))
    img.paste((220, 20, 20), (0, 0, 224, 224))
    replies = {}
    for side in ("left", "right"):
        q = f"This picture has two halves. What colour is the {side} half? Answer with one word."
        g = model.generate([Request(prompt=q, params={"messages": [{"role": "user", "content": q}], "images": [img],
                                                      "max_tokens": 16, "temperature": 0.0})])[0].completions[0]
        replies[side] = (g.text or g.error or "").strip()[:40]
    ok = "red" in replies["left"].lower() and "blue" in replies["right"].lower()
    return ok, replies

## Gate check (a few seconds)

All nine are gated; this confirms access before the long run starts.

In [ ]:
GATED = []
for repo, *_ in MODELS:
    try:
        hf_hub_download(repo, "config.json", token=os.environ.get("HF_TOKEN"))
    except Exception as e:
        GATED.append(repo)
        print(f"  no access: {repo} ({type(e).__name__}) -> accept at https://huggingface.co/{repo}")
print("all nine accessible" if not GATED else f"{len(GATED)} not accessible: they will be SKIP")

## The sweep

In [ ]:
PROBES_JSON = json.dumps(PROBES)
clear_caches()
print(f"disk free before the sweep: {free_disk_gb():.0f} GB")
for repo, arch, need, is_vision in MODELS:
    print(f"\n=== {repo} ({arch}, needs ~{need} GB) ===")
    try:
        archs, model_type = config_archs(repo)
        hit = [a for a in archs if a in REGISTERED]
        record(repo, "registry", "declared" if hit else "absent", f"model_type={model_type} architectures={archs}")
    except Exception as e:
        record(repo, "registry", "unknown", f"{type(e).__name__}: {e}")
    if repo in GATED:
        record(repo, "offline", "SKIP", "licence not accepted for this HF token"); continue
    if GPU_GB < need:
        record(repo, "offline", "SKIP", f"needs ~{need} GB, GPU has {GPU_GB:.0f}"); continue
    try:
        disk_need = download_gb(repo) + 5        # the real files, plus working room
    except Exception:
        disk_need = need + 10                    # the estimate, if the Hub doesn't list sizes
    if free_disk_gb() < disk_need:
        delete_weights(repo); clear_caches()      # a half-finished download, stale caches
        if free_disk_gb() < disk_need:
            record(repo, "offline", "SKIP", f"needs ~{disk_need:.0f} GB of disk to download, {free_disk_gb():.0f} GB free"); continue

    # 1. offline vllm:, in its own process
    out, log = f"/tmp/vllm_one_{repo.split('/')[-1]}.json", f"/tmp/vllm_one_{repo.split('/')[-1]}.log"
    env = {**os.environ, "PYTHONPATH": f"{REPO}/src"}
    def offline(engine):
        if os.path.exists(out): os.remove(out)
        with open(log, "w") as lf:
            rc = subprocess.run([sys.executable, WORKER_PY, repo, out, PROBES_JSON, json.dumps(engine)], stdout=lf,
                                stderr=subprocess.STDOUT, env=env, timeout=LOAD_TIMEOUT_S + 1800).returncode
        res = json.load(open(out)) if os.path.exists(out) else {"error": f"worker exited {rc} without a result"}
        if "error" in res:
            res["cause"], res["line"] = classify(res.get("traceback", "") + "\n" + open(log, errors="ignore").read()
                                                 + "\n" + res["error"])
        return res
    res, label = offline({}), ""
    if "error" in res and res["cause"] in NATIVE_ONLY and TRANSFORMERS_BACKEND_RETRY:
        record(repo, "offline", "RETRY", f"{res['cause']}: {res['line']}")
        wait_memory_back()
        res, label = offline({"model_impl": "transformers"}), TF_LABEL
    if "error" in res:
        record(repo, "offline", "FAIL", f"{res['cause']}: {res['line']}{label}")
    else:
        returned = res["free_after_unload_gb"] - res["free_before_unload_gb"]
        record(repo, "offline", "PASS", f"loaded in {res['load_s']}s; chat template applied: {res['chat_template']}; "
               f"unload returned {returned:.1f} GB ({res['free_after_unload_gb']} of {res['total_gb']} GB free){label}")
        record(repo, "offline text", verdict(res["right"]),
               f"{res['right']}/{len(PROBES)} :: " + " | ".join(f"{k}={v!r}" for k, v in res["answers"].items()))
    wait_memory_back()

    # 2. vllm serve, through api:
    slog = f"/tmp/vllm_serve_{repo.split('/')[-1]}.log"
    t0, proc, slabel = time.time(), start_server(repo, slog), ""
    try:
        healthy = wait_healthy(proc, LOAD_TIMEOUT_S)
        if not healthy:
            text = open(slog, errors="ignore").read()
            cause, line = classify(text) if proc.poll() is not None else ("timeout", f"not healthy after {LOAD_TIMEOUT_S}s")
            if cause in NATIVE_ONLY and TRANSFORMERS_BACKEND_RETRY:
                record(repo, "serve", "RETRY", f"{cause}: {line}")
                stop(proc)
                t0, proc, slabel = time.time(), start_server(repo, slog, ("--model-impl", "transformers")), TF_LABEL
                healthy = wait_healthy(proc, LOAD_TIMEOUT_S)
                if not healthy:
                    text = open(slog, errors="ignore").read()
                    cause, line = classify(text) if proc.poll() is not None else ("timeout", f"not healthy after {LOAD_TIMEOUT_S}s")
        if not healthy:
            record(repo, "serve", "FAIL", f"{cause}: {line}{slabel}")
            print("\n".join(text.splitlines()[-12:]))
        else:
            record(repo, "serve", "PASS", f"healthy after {time.time()-t0:.0f}s{slabel}")
            m = APIModel(repo, api_base=f"http://127.0.0.1:{PORT}/v1", api_key="EMPTY", name=f"api:vllm/{repo}",
                         server="vllm")
            right, answers = probes(m)
            record(repo, "serve text", verdict(right),
                   f"{right}/{len(PROBES)} :: " + " | ".join(f"{k}={v!r}" for k, v in answers.items()))
            if is_vision:
                reply = image_check(m)
                record(repo, "image", "PASS" if "red" in reply.lower() else "FAIL", reply)
                ok, replies = layout_check(m)
                record(repo, "layout", "PASS" if ok else "FAIL", f"left={replies['left']!r} right={replies['right']!r}")
    except Exception as e:
        record(repo, "harness", "ERROR", f"{type(e).__name__}: {e}")
    finally:
        stop(proc)
        if DELETE_AFTER:
            delete_weights(repo)

## Results

In [ ]:
import pandas as pd
df = pd.DataFrame(RESULTS)
table = df.pivot_table(index="model", columns="check", values="outcome", aggfunc="last")   # last: a retry's outcome.reindex([m[0] for m in MODELS])
display(table[[c for c in ("registry", "offline", "offline text", "serve", "serve text", "image", "layout", "harness") if c in table]])
display(df[df.outcome.isin(["FAIL", "ERROR", "PARTIAL", "absent", "unknown"])][["model", "check", "outcome", "detail"]])
with open("vllm_cohere_matrix.json", "w") as f:
    json.dump({"versions": VERSIONS, "gpu": GPU_NAME, "gpu_gb": round(GPU_GB, 1),
               "date": time.strftime("%Y-%m-%d"), "results": RESULTS}, f, indent=1, ensure_ascii=False)
print("wrote vllm_cohere_matrix.json:", VERSIONS, GPU_NAME)

### Reading it

- **offline FAIL `removed from vLLM`:** vLLM names the last version that had the architecture. Expected for Aya Vision (v0.24.0): it's a removal, not a missing implementation.
- **`kernel build` FAIL ("FlashInfer requires GPUs with sm75 or higher"):** FlashInfer couldn't compile for this GPU with its CUDA toolkit (SM 12.x needs CUDA ≥ 12.9). The launch advice printed in the install cell should have switched vLLM to its own sampler; if it's empty, `BRANCH` predates it.
- **offline PASS with `chat template applied: False`:** the named-template fix is missing, so check `BRANCH`.
- **`unload returned`** should be close to the model's footprint (weights plus KV cache). A small number with little memory free means the teardown leak (#54) is back.
- **serve FAIL `out of memory` right after an offline PASS:** memory from the offline run was not returned (the same leak). The offline worker is a separate process, so it should never happen here.
- **text PARTIAL or FAIL on a model that loads:** a quality result, comparable with the `hf:` and SGLang columns.
- **layout FAIL with image PASS:** the model sees the image but not where things are in it; compare with the SGLang and `hf:` columns.
- **SKIP on disk:** the line says how much was needed and free; restart the runtime (fresh disk) and run again.
- **Send back** `vllm_cohere_matrix.json` (in the Files pane) or the two tables above.